In [1]:
import pandas as pd
import numpy as np
import random
import json
import os
import warnings
warnings.filterwarnings("ignore")

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (classification_report, accuracy_score,
                              mean_absolute_error, confusion_matrix)
from scipy.sparse import hstack, issparse
import scipy.sparse as sp

In [2]:
data = pd.read_csv('Sample_arvyax_reflective_dataset.xlsx - Dataset_120.csv')

In [3]:


print(" ── SECTION 2 — FEATURE ENGINEERING & PREPROCESSING (UPDATED) ──")

# 1. Load the Test Data (the 120 samples)
# Adjust the filename/path as needed for your environment
test_file_path = "arvyax_test_inputs_120.xlsx - Sheet1.csv"
test_df = pd.read_csv(test_file_path)

# 2. Text Preprocessing Function
def clean_text(text):
    if pd.isna(text) or str(text).strip() == "":
        return "missing reflection"
    return str(text).strip().lower()

# Clean both datasets
data["journal_text_clean"] = data["journal_text"].apply(clean_text)
test_df["journal_text_clean"] = test_df["journal_text"].apply(clean_text)

# 3. TF-IDF Processing
tfidf = TfidfVectorizer(
    ngram_range=(1, 2), 
    max_features=1000, 
    sublinear_tf=True, 
    strip_accents="unicode",
)

# FIT on training data, TRANSFORM both
text_features_train = tfidf.fit_transform(data["journal_text_clean"])
text_features_test = tfidf.transform(test_df["journal_text_clean"])

print(f"[TEXT] Vocab size: {len(tfidf.get_feature_names_out())}")

# 4. Metadata Preprocessing
numerical_cols = ["duration_min", "sleep_hours", "energy_level", "stress_level"]
categorical_cols = ["ambience_type", "time_of_day", "previous_day_mood", 
                    "face_emotion_hint", "reflection_quality"]

metadata_preprocessor = ColumnTransformer(transformers=[
    ("num", Pipeline([
        ("impute", SimpleImputer(strategy="median")), 
        ("scale", StandardScaler()), 
    ]), numerical_cols),
    ("cat", Pipeline([
        ("impute", SimpleImputer(strategy="constant", fill_value="missing")),
        ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]), categorical_cols),
])

# FIT on training data, TRANSFORM both
meta_features_train = metadata_preprocessor.fit_transform(data[numerical_cols + categorical_cols])
meta_features_test = metadata_preprocessor.transform(test_df[numerical_cols + categorical_cols])

# 5. Combine Features
X_train_full = hstack([text_features_train, meta_features_train])
X_test_final = hstack([text_features_test, meta_features_test]) # This is for your final submission

print(f"[COMBINED] Training Matrix: {X_train_full.shape}")
print(f"[COMBINED] Test Matrix (120 samples): {X_test_final.shape}")

# 6. Target Encoding (Labels)
le = LabelEncoder()
y_state_encoded = le.fit_transform(data["emotional_state"])
state_labels = le.classes_
y_intensity = data["intensity"].values.astype(float)

# 7. Train/Validation Split (on the LABELED data)
X_train, X_val, y_s_train, y_s_val, y_i_train, y_i_val = train_test_split(
    X_train_full, y_state_encoded, y_intensity,
    test_size=0.2, random_state=42, stratify=y_state_encoded
)

print(f"\n[DONE] Train: {X_train.shape[0]} | Val: {X_val.shape[0]} | Final Test: {X_test_final.shape[0]}")

 ── SECTION 2 — FEATURE ENGINEERING & PREPROCESSING (UPDATED) ──
[TEXT] Vocab size: 1000
[COMBINED] Training Matrix: (1200, 1031)
[COMBINED] Test Matrix (120 samples): (120, 1031)

[DONE] Train: 960 | Val: 240 | Final Test: 120


In [28]:
# PART 1 — EMOTIONAL STATE PREDICTION (CLASSIFICATION)
print("  PART 1 — EMOTIONAL STATE PREDICTION")

# Convert to dense for GradientBoosting (it doesn't support sparse directly)
X_train_dense = X_train.toarray()
X_val_dense = X_val.toarray()

state_model = GradientBoostingClassifier(
    n_estimators=150,     
    learning_rate=0.08,  
    max_depth=4,          
    subsample=0.8,     
    random_state=42,
)
state_model.fit(X_train_dense, y_s_train)

y_s_pred = state_model.predict(X_val_dense)
acc = accuracy_score(y_s_val, y_s_pred)
print(f"\n[PART 1] Validation Accuracy: {acc:.4f}")
print("\n[PART 1] Detailed Classification Report:")
print(classification_report(y_s_val, y_s_pred, target_names=state_labels))


  PART 1 — EMOTIONAL STATE PREDICTION

[PART 1] Validation Accuracy: 0.6833

[PART 1] Detailed Classification Report:
              precision    recall  f1-score   support

        calm       0.74      0.67      0.71        43
     focused       0.60      0.69      0.64        39
       mixed       0.68      0.61      0.64        38
     neutral       0.75      0.68      0.71        40
 overwhelmed       0.64      0.76      0.70        38
    restless       0.71      0.69      0.70        42

    accuracy                           0.68       240
   macro avg       0.69      0.68      0.68       240
weighted avg       0.69      0.68      0.68       240



In [ ]:
import numpy as np
from sklearn.metrics import mean_absolute_error
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import cross_val_score

# ── PART 2 — INTENSITY PREDICTION (REGRESSION) ──────────────────────────────

intensity_model = GradientBoostingRegressor(
    n_estimators=300,          # more trees → smoother predictions
    learning_rate=0.05,        # lower LR + more trees is usually better
    max_depth=3,               # shallower → less overfit on noisy labels
    subsample=0.8,
    min_samples_leaf=5,        # prevents fitting tiny noisy groups
    loss='huber',
    alpha=0.9,                 # huber threshold — keep high for label noise
    random_state=42,
)
intensity_model.fit(X_train_dense, y_i_train)

# ── Raw prediction ───────────────────────────────────────────────────────────
y_i_pred_raw = intensity_model.predict(X_val_dense)
y_i_pred = np.clip(np.round(y_i_pred_raw), 1, 5).astype(int)

# ── PART 4 — BETTER CONFIDENCE via estimator spread ─────────────────────────
staged = np.array([
    np.clip(est.predict(X_val_dense), 1, 5)
    for est in intensity_model.estimators_[:, 0]
])  # shape: (n_estimators, n_samples)

pred_std = staged.std(axis=0)          # spread across trees = uncertainty proxy

max_std = pred_std.max() if pred_std.max() > 0 else 1.0
confidence = np.clip(1 - (pred_std / max_std), 0.0, 1.0)

# Threshold: flag uncertain if confidence < 0.5 OR prediction far from integer
distance_from_int = np.abs(y_i_pred_raw - np.round(y_i_pred_raw))
uncertain_flag = ((confidence < 0.5) | (distance_from_int > 0.4)).astype(int)

# ── Evaluation ───────────────────────────────────────────────────────────────
mae = mean_absolute_error(y_i_val, y_i_pred)

# Cross-val MAE on training set (more reliable than single split)
cv_scores = cross_val_score(
    intensity_model, X_train_dense, y_i_train,
    scoring='neg_mean_absolute_error', cv=5
)
cv_mae = -cv_scores.mean()

print(f"[PART 2] MAE (val, rounded):        {mae:.4f}")
print(f"[PART 2] CV MAE (train, 5-fold):    {cv_mae:.4f}")
print(f"[PART 2] Raw predictions sample:    {y_i_pred_raw[:5].round(2)}")
print(f"[PART 2] Rounded predictions:       {y_i_pred[:5]}")
print(f"[PART 4] Pred std (uncertainty):    {pred_std[:5].round(3)}")
print(f"[PART 4] Confidence scores:         {confidence[:5].round(2)}")
print(f"[PART 4] Uncertain flags:           {uncertain_flag[:5]}")
print(f"[PART 4] Total uncertain cases:     {uncertain_flag.sum()} / {len(uncertain_flag)}")
print(f"[PART 4] Uncertain rate:            {uncertain_flag.mean()*100:.1f}%")

[PART 2] MAE (val, rounded):        1.2208
[PART 2] CV MAE (train, 5-fold):    1.2355
[PART 2] Raw predictions sample:    [3.04 2.41 3.81 3.03 3.7 ]
[PART 2] Rounded predictions:       [3 2 4 3 4]
[PART 4] Pred std (uncertainty):    [0.017 0.008 0.009 0.092 0.028]
[PART 4] Confidence scores:         [0.82 0.92 0.91 0.05 0.71]
[PART 4] Uncertain flags:           [0 1 0 1 0]
[PART 4] Total uncertain cases:     55 / 240
[PART 4] Uncertain rate:            22.9%


In [16]:
import random

class DecisionEngine:
    
    OPENERS = {
        "calm":      ["You seem grounded right now.", "There's a quiet steadiness coming through."],
        "anxious":   ["It sounds like your mind is running fast.", "There's some restlessness in what you've shared."],
        "restless":  ["Something feels unsettled.", "You seem like you're between things right now."],
        "sad":       ["It sounds like today has been heavy.", "There's a weight in what you've written."],
        "energized": ["You've got some real momentum right now.", "There's good energy in what you've shared."],
        "tired":     ["Your body seems like it's asking for rest.", "It sounds like today's taken a lot out of you."],
        "focused":   ["You seem clear-headed right now.", "There's good focus coming through."],
        "mixed":     ["Your signals are a bit mixed today.", "Today seems to have a few layers to it."],
    }
    
    ACTION_TEMPLATES = {
        "box_breathing":   {
            "now":           "Take 4 minutes right now — breathe in for 4, hold for 4, out for 4. Just one round.",
            "within_15_min": "In the next 15 minutes, find a quiet spot and do 3–4 rounds of box breathing.",
        },
        "journaling":      {
            "now":           "Take 5 minutes to write whatever's on your mind — no structure needed.",
            "later_today":   "Tonight, try writing for 10 minutes. Don't edit. Just let it out.",
            "tonight":       "Before bed, jot down 3 things from today — good, bad, or just notable.",
        },
        "grounding":       {
            "now":           "Pause for a moment. Name 5 things you can see, 4 you can touch, 3 you can hear.",
            "within_15_min": "Step outside or to a window soon — a few minutes of fresh air and ground beneath you.",
        },
        "deep_work":       {
            "now":           "You're in a good window for focused work. Start with your most important task.",
            "within_15_min": "In a bit, clear your notifications and block out 60–90 minutes of deep focus.",
            "later_today":   "This afternoon could be a good window for deep work once your energy settles.",
        },
        "rest":            {
            "now":           "Give yourself permission to rest — even 15–20 minutes makes a difference.",
            "tonight":       "Prioritize sleep tonight. Aim to be in bed an hour earlier than usual.",
            "tomorrow_morning": "Tomorrow morning, take it slow. Don't rush into tasks right away.",
        },
        "movement":        {
            "now":           "Get up and move — a 10-minute walk, some stretching, anything that gets you out of your head.",
            "within_15_min": "Soon, step away from your screen and move your body for a bit.",
            "later_today":   "This afternoon, try to fit in a walk or some light movement.",
        },
        "sound_therapy":   {
            "now":           "Put on something calming — lo-fi, nature sounds, or silence. Let it just be background.",
            "tonight":       "Tonight, wind down with some soft music or ambient sound instead of screens.",
        },
        "light_planning":  {
            "now":           "Spend 5 minutes writing out the 2–3 things that actually matter today. Just those.",
            "within_15_min": "Before diving into work, take a few minutes to map out the day simply.",
        },
        "yoga":            {
            "now":           "Even 10 minutes of gentle stretching or a short yoga flow can reset your nervous system.",
            "later_today":   "Try to carve out time this afternoon for some yoga or stretching.",
        },
        "pause":           {
            "now":           "Before deciding anything, just pause. Sit with where you are for a minute.",
            "within_15_min": "Give yourself a few minutes before jumping into anything. You don't need to rush.",
        },
    }

    INTENSITY_CLOSERS = {
        1: "Even small steps count.",
        2: "Nothing drastic — just a small nudge in the right direction.",
        3: "This is a good moment to make one clear choice for yourself.",
        4: "Right now, one intentional action can shift things meaningfully.",
        5: "This feels important. Give yourself what you need — don't push through.",
    }

    UNCERTAINTY_HEDGES = [
        "I'm reading between some mixed lines here, so take this as a gentle suggestion —",
        "Your signals are a bit nuanced today, so this is more of a nudge than a prescription —",
        "I'm not entirely certain what's going on, but if I had to offer one thing —",
    ]

    def decide(self, state, intensity, stress, energy, time_of_day, uncertain_flag=0):
        """
        Multi-layer decision engine.
        Returns (action, timing)
        """
        state = state.lower()
        
        # ── LAYER 1: Time-of-day baseline ──────────────────────────────────
        if time_of_day == "morning":
            base_action, base_timing = "light_planning", "now"
        elif time_of_day == "afternoon":
            base_action, base_timing = "deep_work", "now"
        elif time_of_day == "evening":
            base_action, base_timing = "journaling", "tonight"
        else:  # night
            base_action, base_timing = "rest", "tonight"

        # ── LAYER 2: Emotional state override ──────────────────────────────
        state_overrides = {
            "anxious":   ("box_breathing", "now"),
            "restless":  ("grounding",     "now"),
            "sad":       ("journaling",    "now"),
            "tired":     ("rest",          "now"),
            "energized": ("deep_work",     "now"),
            "focused":   ("deep_work",     "now"),
        }
        action, timing = state_overrides.get(state, (base_action, base_timing))

        # ── LAYER 3: Stress × energy matrix ────────────────────────────────
        # High stress overrides toward decompression
        if stress >= 4 and energy >= 3:
            action, timing = "movement", "within_15_min"
        elif stress >= 4 and energy < 3:
            action, timing = "box_breathing", "now"

        # Low energy → rest regardless of state
        if energy <= 2 and time_of_day in ("evening", "night"):
            action, timing = "rest", "tonight"
        elif energy <= 1:
            action, timing = "rest", "now"

        # ── LAYER 4: Intensity amplifier ───────────────────────────────────
        # High intensity + negative states → immediate grounding
        negative_states = {"anxious", "sad", "restless"}
        if intensity >= 4 and state in negative_states:
            action = "grounding" if state == "restless" else action
            timing = "now"
        
        # Low intensity → shift timing to "later" if urgent action was assigned
        if intensity <= 2 and timing == "now" and action not in ("rest",):
            timing = "within_15_min"

        # ── LAYER 5: Uncertainty softening (refined) ────────────────────────
        if uncertain_flag == 1:
            if stress < 3 and intensity <= 2:
                # Genuinely unclear + low stakes → soft pause
                action, timing = "pause", "now"
            elif timing == "now" and stress < 4:
                # Uncertain but moderate → nudge timing back slightly
                timing = "within_15_min"
            # High stress uncertain cases: keep the action, just soften the message (handled in generate_message)

        return action, timing

    def generate_message(self, state, action, timing, intensity, uncertain_flag=0):
        """
        Generates a warm, contextual, human-sounding recommendation message.
        """
        state_key = state.lower() if state.lower() in self.OPENERS else "mixed"
        
        # ── Choose opener based on state ──────────────────────────────────
        opener = random.choice(self.OPENERS[state_key])
        
        # ── Get action instruction for the given timing ────────────────────
        action_map = self.ACTION_TEMPLATES.get(action, {})
        
        # Fallback timing if exact key missing
        instruction = (
            action_map.get(timing)
            or action_map.get("now")
            or f"Consider {action.replace('_', ' ')} {timing.replace('_', ' ')}."
        )

        # ── Intensity-aware closer ─────────────────────────────────────────
        closer = self.INTENSITY_CLOSERS.get(intensity, self.INTENSITY_CLOSERS[3])

        # ── Uncertainty hedge (replaces generic "complex vibes" header) ────
        if uncertain_flag == 1:
            hedge = random.choice(self.UNCERTAINTY_HEDGES)
            # Hedge goes BETWEEN opener and instruction, not as a preamble
            full_msg = f"{opener} {hedge} {instruction} {closer}"
        else:
            full_msg = f"{opener} {instruction} {closer}"

        return full_msg
    # Create the instance that the inference pipeline is looking for
engine = DecisionEngine()

In [17]:

# PART 4 — UNCERTAINTY MODELING
print("  PART 4 — UNCERTAINTY MODELING")

def compute_uncertainty(state_probs, intensity_raw, text, metadata_row):
    reasons = []

    # Signal 1: Classification confidence
    max_prob = float(np.max(state_probs))
    if max_prob < 0.45:
        reasons.append(f"low_class_confidence ({max_prob:.2f})")

    # Signal 2: Intensity borderline
    intensity_dist = abs(intensity_raw - round(intensity_raw))
    if intensity_dist > 0.4:
        reasons.append(f"borderline_intensity ({intensity_raw:.2f})")

    # Signal 3: Very short text
    word_count = len(str(text).split())
    if word_count <= 3:
        reasons.append(f"short_text ({word_count} words)")

    # Signal 4: Missing metadata
    missing_count = sum(1 for v in metadata_row if pd.isna(v))
    if missing_count >= 2:
        reasons.append(f"missing_metadata ({missing_count} fields)")

    uncertain_flag = 1 if len(reasons) > 0 else 0
    return max_prob, uncertain_flag, reasons

# Demonstrate
sample_probs = state_model.predict_proba(X_val_dense[:3])
for i in range(3):
    row = data.iloc[i]
    conf, flag, reasons = compute_uncertainty(
        state_probs=sample_probs[i],
        intensity_raw=y_i_pred_raw[i] if i < len(y_i_pred_raw) else 3.0,
        text=row["journal_text"],
        metadata_row=[row[c] for c in numerical_cols],
    )
    print(f"[PART 4] Sample {i+1}: confidence={conf:.3f}, uncertain={flag}, reasons={reasons}")


  PART 4 — UNCERTAINTY MODELING
[PART 4] Sample 1: confidence=0.528, uncertain=0, reasons=[]
[PART 4] Sample 2: confidence=0.321, uncertain=1, reasons=['low_class_confidence (0.32)', 'borderline_intensity (2.41)']
[PART 4] Sample 3: confidence=0.735, uncertain=0, reasons=[]


In [18]:

# INFERENCE PIPELINE — Full run_inference function
print("  INFERENCE PIPELINE — run_inference()")

def run_inference(df, state_model, intensity_model, meta_preprocessor,
                  tfidf, state_labels, le, engine):
    results = []

    # ── Step 1: Text Features ──────────────────────────────────────────────
    texts_clean = df["journal_text"].apply(clean_text).tolist()
    text_feats = tfidf.transform(texts_clean)          # uses fitted TF-IDF

    # ── Step 2: Metadata Features ─────────────────────────────────────────
    meta_cols = numerical_cols + categorical_cols
    # Fill in missing categorical columns that might not be in the input
    for col in meta_cols:
        if col not in df.columns:
            df[col] = np.nan
    meta_feats = meta_preprocessor.transform(df[meta_cols])

    # ── Step 3: Combine & Predict ─────────────────────────────────────────
    X_combined = hstack([text_feats, meta_feats]).toarray()
    state_probs_all = state_model.predict_proba(X_combined)
    state_preds_enc = state_model.predict(X_combined)
    state_preds = le.inverse_transform(state_preds_enc)
    intensity_raw_all = intensity_model.predict(X_combined)
    intensity_preds = np.clip(np.round(intensity_raw_all), 1, 5).astype(int)

    # ── Step 4: Per-Row Decision + Uncertainty ────────────────────────────
    for idx, row in df.iterrows():
        i = list(df.index).index(idx)   # position in batch
        pred_state = state_preds[i]
        pred_intensity = int(intensity_preds[i])
        int_raw = float(intensity_raw_all[i])
        probs = state_probs_all[i]

        # Uncertainty
        stress = row.get("stress_level", np.nan)
        energy = row.get("energy_level", np.nan)
        stress = float(stress) if pd.notna(stress) else 3.0   # impute missing
        energy = float(energy) if pd.notna(energy) else 3.0

        conf, unc_flag, reasons = compute_uncertainty(
            state_probs=probs,
            intensity_raw=int_raw,
            text=row.get("journal_text", ""),
            metadata_row=[row.get(c, np.nan) for c in numerical_cols],
        )

        # Decision
        time_of_day = row.get("time_of_day", "morning")
        if pd.isna(time_of_day):
            time_of_day = "morning"
        what_to_do, when_to_do = engine.decide(
            state=pred_state,
            intensity=pred_intensity,
            stress=stress,
            energy=energy,
            time_of_day=str(time_of_day),
        )

        # Supportive message
        message = engine.generate_message(pred_state, what_to_do, when_to_do, pred_intensity)

        results.append({
            "id": row.get("id", f"row_{idx}"),
            "journal_text": row.get("journal_text", ""),
            "predicted_state": pred_state,
            "predicted_intensity": pred_intensity,
            "confidence": round(conf, 4),
            "uncertain_flag": unc_flag,
            "uncertainty_reasons": "; ".join(reasons) if reasons else "none",
            "what_to_do": what_to_do,
            "when_to_do": when_to_do,
            "supportive_message": message,
            # Ground truth (if available)
            "true_state": row.get("emotional_state", "unknown"),
            "true_intensity": row.get("intensity", -1),
        })

    return pd.DataFrame(results)


# Run on validation set
val_df = data.iloc[X_val.shape[0]:].reset_index(drop=True) \
    if len(data) > X_val.shape[0] else data.sample(20, random_state=42).reset_index(drop=True)

# Actually run on full dataset for output
all_results = run_inference(
    data.copy(), state_model, intensity_model,
    metadata_preprocessor, tfidf, state_labels, le, engine
)

print(f"[INFERENCE] Ran on {len(all_results)} samples")
print(all_results[["id", "predicted_state", "predicted_intensity",
                    "confidence", "uncertain_flag", "what_to_do",
                    "when_to_do"]].head(8).to_string(index=False))

  INFERENCE PIPELINE — run_inference()
[INFERENCE] Ran on 1200 samples
 id predicted_state  predicted_intensity  confidence  uncertain_flag     what_to_do    when_to_do
  1         focused                    3      0.9543               0      deep_work           now
  2        restless                    3      0.9171               0           rest       tonight
  3            calm                    3      0.8076               0           rest       tonight
  4         neutral                    3      0.7452               1       movement within_15_min
  5     overwhelmed                    3      0.8918               0       movement within_15_min
  6            calm                    3      0.9179               0 light_planning           now
  7         neutral                    2      0.9633               0  box_breathing within_15_min
  8        restless                    4      0.9599               0      grounding           now


In [20]:
# PART 5 — FEATURE UNDERSTANDING
print("  PART 5 — FEATURE IMPORTANCE ANALYSIS")
cat_names = (metadata_preprocessor
             .named_transformers_["cat"]["ohe"]
             .get_feature_names_out(categorical_cols).tolist())
num_names = numerical_cols
meta_feature_names = num_names + cat_names

n_text_feats = text_features.shape[1]
n_meta_feats = X_meta.shape[1]
text_feat_names = [f"tfidf_{w}" for w in tfidf.get_feature_names_out()]

all_feature_names = text_feat_names + meta_feature_names

importances = state_model.feature_importances_

min_len = min(len(importances), len(all_feature_names))
imp_df = pd.DataFrame({
    "feature": all_feature_names[:min_len],
    "importance": importances[:min_len],
    "type": ["text" if i < n_text_feats else "metadata"
             for i in range(min_len)]
})
imp_df = imp_df.sort_values("importance", ascending=False)

print("\n[PART 5] Top 15 Most Important Features (Emotional State Model):")
print(imp_df.head(15).to_string(index=False))

text_imp_total = imp_df[imp_df["type"] == "text"]["importance"].sum()
meta_imp_total = imp_df[imp_df["type"] == "metadata"]["importance"].sum()
total = text_imp_total + meta_imp_total
print(f"\n[PART 5] Text features contribute:     {text_imp_total/total*100:.1f}%")
print(f"Metadata features contribute: {meta_imp_total/total*100:.1f}%")
print("INSIGHT: Text drives STATE detection; metadata drives INTENSITY.")

  PART 5 — FEATURE IMPORTANCE ANALYSIS

[PART 5] Top 15 Most Important Features (Emotional State Model):
             feature  importance     type
       tfidf_but not    0.027045     text
       tfidf_nothing    0.024639     text
       tfidf_lighter    0.020248     text
        stress_level    0.018114 metadata
       tfidf_drained    0.018023     text
     tfidf_organized    0.017258     text
         tfidf_tasks    0.016910     text
        duration_min    0.016454 metadata
       tfidf_between    0.014535     text
         tfidf_quiet    0.014309     text
tfidf_still mentally    0.014228     text
   tfidf_concentrate    0.013858     text
          tfidf_less    0.013672     text
         sleep_hours    0.012737 metadata
     tfidf_felt more    0.012578     text

[PART 5] Text features contribute:     85.9%
Metadata features contribute: 14.1%
INSIGHT: Text drives STATE detection; metadata drives INTENSITY.


In [22]:
# PART 6 — ABLATION STUDY
print("  PART 6 — ABLATION STUDY")

X_train_dense_full = X_train.toarray()
X_val_dense_full = X_val.toarray()

n_text = n_text_feats      # first n_text_feats columns are TF-IDF
n_meta = X_train_dense_full.shape[1] - n_text

X_train_txt_only = X_train_dense_full[:, :n_text]
X_val_txt_only   = X_val_dense_full[:, :n_text]
X_train_meta_only = X_train_dense_full[:, n_text:]
X_val_meta_only   = X_val_dense_full[:, n_text:]

def quick_model(X_tr, y_tr, X_va, y_va, task="clf"):
    if task == "clf":
        m = GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=42)
        m.fit(X_tr, y_tr)
        return accuracy_score(y_va, m.predict(X_va))
    else:
        m = GradientBoostingRegressor(n_estimators=100, max_depth=3, random_state=42)
        m.fit(X_tr, y_tr)
        preds = np.clip(np.round(m.predict(X_va)), 1, 5)
        return mean_absolute_error(y_va, preds)

print("[PART 6] Training ablation variants...")

# State prediction ablation
acc_txt_only   = quick_model(X_train_txt_only, y_s_train, X_val_txt_only, y_s_val, "clf")
acc_meta_only  = quick_model(X_train_meta_only, y_s_train, X_val_meta_only, y_s_val, "clf")
acc_hybrid     = accuracy_score(y_s_val, state_model.predict(X_val_dense_full))

# Intensity ablation
mae_txt_only   = quick_model(X_train_txt_only, y_i_train, X_val_txt_only, y_i_val, "reg")
mae_meta_only  = quick_model(X_train_meta_only, y_i_train, X_val_meta_only, y_i_val, "reg")
mae_hybrid     = mean_absolute_error(y_i_val, np.clip(np.round(intensity_model.predict(X_val_dense_full)), 1, 5))

print("\n╔══════════════════════════════════════════════════════╗")
print("║          ABLATION STUDY RESULTS                      ║")
print("╠════════════════════╦═════════════╦════════════════════╣")
print("║ Configuration      ║ State Acc ↑ ║ Intensity MAE ↓    ║")
print("╠════════════════════╬═════════════╬════════════════════╣")
print(f"║ Text-Only          ║  {acc_txt_only:.4f}      ║  {mae_txt_only:.4f}             ║")
print(f"║ Metadata-Only      ║  {acc_meta_only:.4f}      ║  {mae_meta_only:.4f}             ║")
print(f"║ Hybrid (Ours) ★    ║  {acc_hybrid:.4f}      ║  {mae_hybrid:.4f}             ║")
print("╚════════════════════╩═════════════╩════════════════════╝")


  PART 6 — ABLATION STUDY
[PART 6] Training ablation variants...

╔══════════════════════════════════════════════════════╗
║          ABLATION STUDY RESULTS                      ║
╠════════════════════╦═════════════╦════════════════════╣
║ Configuration      ║ State Acc ↑ ║ Intensity MAE ↓    ║
╠════════════════════╬═════════════╬════════════════════╣
║ Text-Only          ║  0.6792      ║  1.2083             ║
║ Metadata-Only      ║  0.1625      ║  1.2750             ║
║ Hybrid (Ours) ★    ║  0.6833      ║  1.2208             ║
╚════════════════════╩═════════════╩════════════════════╝


In [24]:

# SAVE predictions.csv
print("  SAVING predictions.csv")

output_cols = [
    "id", "predicted_state", "predicted_intensity",
    "confidence", "uncertain_flag",
    "what_to_do", "when_to_do",
    "supportive_message",
    "uncertainty_reasons",
]
all_results[output_cols].to_csv("predictions.csv", index=False)
print(f"[OUTPUT] predictions.csv saved with {len(all_results)} rows")
print(all_results[output_cols].head(5).to_string(index=False))



  SAVING predictions.csv
[OUTPUT] predictions.csv saved with 1200 rows
 id predicted_state  predicted_intensity  confidence  uncertain_flag what_to_do    when_to_do                                                                                                                                                                 supportive_message         uncertainty_reasons
  1         focused                    3      0.9543               0  deep_work           now     There's good focus coming through. You're in a good window for focused work. Start with your most important task. This is a good moment to make one clear choice for yourself.                        none
  2        restless                    3      0.9171               0       rest       tonight You seem like you're between things right now. Prioritize sleep tonight. Aim to be in bed an hour earlier than usual. This is a good moment to make one clear choice for yourself.                        none
  3            calm       

In [ ]:

# SAVE MODELS (for Flask API)
import pickle

artifacts = {
    "state_model": state_model,
    "intensity_model": intensity_model,
    "metadata_preprocessor": metadata_preprocessor,
    "tfidf": tfidf,
    "state_labels": state_labels,
    "le": le,
    "numerical_cols": numerical_cols,
    "categorical_cols": categorical_cols,
}
with open("model_artifacts.pkl", "wb") as f:
    pickle.dump(artifacts, f)
print("[OUTPUT] model_artifacts.pkl saved")

print("  PIPELINE COMPLETE — ALL PARTS DONE")


[OUTPUT] model_artifacts.pkl saved

  ✅ PIPELINE COMPLETE — ALL PARTS DONE
